# Digital Signatures

Goal: Similar to the symmetric MAC tag, create an authentication message:
- $Gen \to (pk, sk)$: the signer creates a public and private key
- $Sign_{sk}(m) \to \sigma$: the signer signs the message with the private key
- $Vrfy_{pk}(m, \sigma) \to b \in \{0, 1\}$: anyone (verifier) verifies the correctness of the signature with the public key  
$Vrfy_{pk}(m, \sigma) = \begin{cases} 1 & \text{if } \sigma \text{ is a valid signature of } m \\ 0 & \text{otherwise} \end{cases}$

Correctness: $Vrfy_{pk}(m, Sign_{sk}(m)) = 1$

## MAC vs Digital Signature

Common goal: Protecting the integrity of the message

MAC: We generate a secret key with everyone we exchange messages with and verify with it

Digital Signature: We share a public *pk* key with everyone and verify with it
- enough to generate one signature that anyone can verify (*publicly verifiable*)
- a $\sigma$ signature can be passed on to a third party for a $m$ message (*transferable*), who can verify it
- it cannot be denied that someone signed a message (*non-repudiation*)

Disadvantage: The signatures are larger than the MAC tags

# Textbook RSA signature scheme

- $Gen$: Choose two large primes $p$ and $q$, compute $N = pq$, $\phi(N) = (p-1)(q-1)$, and choose $e$ such that $1 < e < \phi(N)$ and $\gcd(e, \phi(N)) = 1$. Compute $d$ such that $ed \equiv 1 \mod \phi(N)$. The public key is $(N, e)$ and the private key is $(N, d)$.
- $Sign_{sk}(m)$: Compute $\sigma = m^d \mod N$.
- $Vrfy_{pk}(m, \sigma)$: Compute $m' = \sigma^e \mod N$ and check if $m' = m$. If they are equal, return 1; otherwise, return 0.


In [0]:
p = next_prime(1234)
q = next_prime(5678)

n = p * q
z = Integers(n)
e = 65537
phi = (p - 1) * (q - 1)
d = pow(e, -1, phi)

print(f"p: {p}, q: {q}, n: {n}, e: {e}, d: {d}, phi: {phi}")

m = z(42)
signature = m ^ d
signature

In [0]:
signature ^ e

## Attacks

### No message attack

For any signature $\sigma$ choose message $m = \sigma^e \mod N$.
This signature is valid for $m$.

Downside: The attacker cannot choose the message (only the signature).

In [0]:
signature = z(13579)
m = signature ^ e
m

### Chosen message attack

The attacker can choose a message $m$ and get the signature $\sigma$ for it if he has access to the signing oracle.
- The attacker can choose $m, m_1$ and $m_2 \equiv m / m_1 \mod N$.
- The attacker queries the signing oracle for $m_1$ and $m_2$ and gets $\sigma_1$ and $\sigma_2$.
- The attacker computes $\sigma = \sigma_1 \cdot \sigma_2 \mod N$.

In [0]:
m = z(365)
m1 = 67
m2 = m / m1

sigma1 = m1 ^ d
sigma2 = m2 ^ d

signature = sigma1 * sigma2
signature

In [0]:
signature ^ e

# Hashed RSA signature scheme

Similar, but instead of signing the message directly, we sign a hash of the message.  
$\implies$ the previous attacks are not possible anymore. Why?

- The first attack fails, because you can only calculate $H(m) = \sigma^e \mod N$, but (as hashes are one-way functions) you cannot calculate $m$ from it.
- The second attack fails, because you can only calculate $H(m_1) \cdot H(m_2) = \sigma_1 \cdot \sigma_2 \mod N$, and you would need $H(m) = H(m_1) \cdot H(m_2)$, but that is generally not true.

# Key card access approaches

## Basic static cards

- The card contains a static identifier (e.g. magnetic stripe cards)
- The reader checks if the identifier is in the database of valid identifiers

An attacker can clone the card by reading the identifier and create counterfeit cards.

## Digital signature based approach

- The card contains a unique private key with a central database having the public keys
- The reader sends a random challenge $c$ to the card (a message to be signed)
- The card signs the challenge with its private key and sends the signature back to the reader
- The reader verifies the signature with the public key in the database

Additional improvements:
- The reader can send also prove its identity to the card.

# Web Certificate System

Early web: no authentication $\implies$ vulnerable to man-in-the-middle attacks

## SSL certificates and Certificate Authorities

- Each website generates its own public/private key pair
- Certificate Authorities (CAs) sign the website's public key, creating a digital certificate
- Browsers ship with trusted CA public keys pre-installed

# Digital Signature Algorithm (1991-2024)

DSA algorithm: [NIST FIPS 186-4](https://csrc.nist.gov/publications/detail/fips/186/4/final)

1. Key Generation:
    - Choose $p,q$ prime numbers such that $p \equiv 1 \pmod{q}$. Let $L,N$ be the length of the two prime numbers in bits. Recommended choices: $(1024, 160), (2048, 224), (2048, 256), (3072, 256)$
    - Let $h \in_R \{2,\dots,p-2\}$
    - Let $g = h^{(p-1)/q} \pmod{p}$ (but other methods exist, see algorithm)
    - The $(p,q,g)$ triple can be public information
    - Let $x \in_R \{1,2,\dots,q-1\}$ and $y = g^x \pmod{p}$
    - $x$ is the private key, $y$ is the public key
2. Signature
    - The signer chooses a hash function. If the hash output is larger than $N = |q|$, we only use the first $N$ bits of the hash output.
    - The signer chooses an arbitrary $k \in_R \{1,2,\dots,q-1\}$
    - Let $r = (g^k \mod{p}) \mod{q}$
    - Let $s = k^{-1} (H(m) + xr) \mod{q}$
    - The signature will be the $(r, s)$ pair.
3. Verification
    - Let $w = s^{-1} \mod{q}$
    - Let $u_1 = H(m)w \mod{q}$
    - Let $u_2 = rw \mod{q}$
    - Let $v = (g^{u_1}y^{u_2} \mod{p}) \mod{q}$
    - The signature is correct if $v = r$
    
**Task**: Show that the above algorithm is correct!

**Solution**: To solve this, we need the following:
From the equations $s = k^{-1}(H(m) + xr) \mod{q}$ and $w = s^{-1} \mod{q} = k(H(m) + xr)^{-1} \mod{q}$, we get $w(H(m) + xr) \mod{q} = k(H(m) + xr)^{-1}(H(m) + xr) \mod{q} = k \mod{q}$. We will use this in the last step.

\begin{array}{ll}
v &= (g^{u_1}y^{u_2} \mod{p}) \mod{q} \\
  &= (g^{H(m)w} y^{rw} \mod{p}) \mod{q} \\
  &= (g^{H(m)w} g^{xrw} \mod{p}) \mod{q} \\
  &= (g^{w(H(m) + rw)} \mod{p}) \mod{q} \\
  &= (g^k \mod{p}) \mod{q} = r
\end{array}

**Attention**: When choosing the value of $k$, be very careful to choose it randomly! 
- OpenSSL, 2008: Due to code optimization in the PRNG (Valgrind), a maximum of 32768 different seed values could be given to the PRNG.
- Playstation 3, 2010: The same $k$ for multiple signatures → acquisition of the private key

## How to encrypt and sign with `GnuPG`?

[`GnuPG`](https://gnupg.org/) is the open-source implementation of OpenPGP ([RFC4880](https://www.ietf.org/rfc/rfc4880.txt))

Public key generation:
- `gpg --full-generate-key`
- `gpg --list-secret-keys`
  ```
  sec   rsa2048/1A4FCCB2EA9CAE58 2023-05-24 [SC]
      3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58
  uid                 [ultimate] Hanyecz Ottó (ELTE Kripto gyakorlat) <ohanyecz@inf.elte.hu>
  ssb   rsa2048/B3801B4006BBB807 2023-05-24 [E]
  ```
- `gpg --armor --export 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58 > pubkey.txt`

Encrypting and decrypting a file:
- `gpg --recipient 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58 --encrypt test.txt`
- `gpg --decrypt test.txt.gpg`

Signing a file:
- `gpg --local-user 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58 --clearsign test.txt`
- `gpg --verify test.txt.gpg`

Obviously, the two can be combined. In this case, during decryption, `gpg` also checks the signature (DO NOT use the same key for encryption and signing):
- `gpg --local-user 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58 --recipient 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58 --armor --sign --encrypt test.txt`
- `gpg --decrypt test.txt.asc`
```
gpg: encrypted with 2048-bit RSA key, ID B3801B4006BBB807, created 2023-05-24
      "Hanyecz Ottó (ELTE Kripto gyakorlat) <ohanyecz@inf.elte.hu>"
This is a test file.
gpg: Signature made 2023. May 24., Wednesday, 10:10:14 CEST
gpg:                using RSA key 3126E047F00A2FF5B6B866EE1A4FCCB2EA9CAE58
gpg: Good signature from "Hanyecz Ottó (ELTE Kripto gyakorlat) <ohanyecz@inf.elte.hu>" [ultimate]
```

How to add the public key?
- `gpg --import someones_pubkey.txt`
- Run `gpg --edit-key <name>` to set the trust level on an interactive prompt